<a href="https://colab.research.google.com/github/cursuri-inf/IPAP/blob/main/saptamana-04/curs04_demonstratii.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cursul 5. Demonstrațiile

Caietul conține demonstrațiile de la cursul 5, „De la un neuron la o rețea”, cu rezultatele salvate. Codul este cel din suportul de curs, în aceeași ordine, iar titlurile de mai jos au numerele capitolelor din suport (§2–§5).

**Cum lucrați:**

1. Deschideți caietul cu butonul „Open in Colab” de mai sus. Colab avertizează că documentul nu a fost creat de Google: caietul vine din depozitul cursului, așa că alegeți „Run anyway”.
2. Înainte să schimbați ceva: File → Save a copy in Drive; lucrați în copie.
3. Rezultatele salvate se văd și fără rulare. Ca să le obțineți din nou, rulați celulele în ordine, de sus în jos, cu Shift+Enter.
4. Celulele „Încercați acum” sunt cele de la curs: schimbați numărul indicat, rulați și comparați cu ce scrie în suport.
5. Notițele dumneavoastră: după fiecare demonstrație aveți o celulă „Notițele mele” (dublu clic pentru editare). Puteți lăsa și comentarii pe marginea din dreapta a oricărei celule: clic dreapta pe celulă, apoi „Add a comment” (sau Ctrl+Alt+M).

Caietul rulează pe procesor; un GPU nu este necesar. Nu se predă și nu se notează.

## 0. Pornire

Bibliotecile, datele (cei 342 de pinguini cu toate măsurătorile) și bucla de antrenare din laboratorul 4. Funcția `pas` este exact cea pe care ați completat-o la laborator; `antreneaza` o cheamă pe fiecare lot, epocă după epocă.

In [1]:
import sys
import torch
from torch import nn
from torch.optim import SGD
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import pandas as pd
from sklearn.model_selection import train_test_split

print("Python ", sys.version.split()[0])
print("PyTorch", torch.__version__)

Python  3.13.18
PyTorch 2.15.0+cpu


In [2]:
URL = ("https://raw.githubusercontent.com/mwaskom/"
       "seaborn-data/master/penguins.csv")
masuri = ["bill_length_mm", "bill_depth_mm",
          "flipper_length_mm", "body_mass_g"]
# 342 de pinguini cu toate măsurătorile
df = pd.read_csv(URL).dropna(subset=masuri)
print(df.shape)

(342, 7)


In [3]:
# funcția pas din laboratorul 4, neschimbată
def pas(model, pierdere_f, opt, zb, yb):
    predictie = model(zb)
    pierdere = pierdere_f(predictie, yb)
    opt.zero_grad()
    pierdere.backward()
    opt.step()

def antreneaza(model, pierdere_f, z, y, rata,
               epoci):
    opt = SGD(model.parameters(), lr=rata)
    loturi = DataLoader(TensorDataset(z, y),
                        batch_size=32, shuffle=True)
    for epoca in range(epoci):
        for zb, yb in loturi:
            pas(model, pierdere_f, opt, zb, yb)

def acuratete(model, z, y):   # 1 când scorul > 0
    with torch.no_grad():
        prezis = (model(z) > 0).float()
    corect = (prezis == y).float().mean().item()
    return round(corect, 3)

## D1. Regresia logistică: un singur neuron (§2)

Un model liniar dă un scor, care poate fi orice număr. Sigmoida îl transformă într-o probabilitate între 0 și 1; scorul 0 dă exact 0,5.

In [4]:
scoruri = torch.tensor([-2.0, 0.0, 1.0, 2.0])
print(torch.sigmoid(scoruri))

tensor([0.1192, 0.5000, 0.7311, 0.8808])


Împărțirea pentru clasificare din cursul 3: 70% antrenare, 15% validare, 15% test, stratificat pe specii. Întrebarea: este pinguinul din specia Gentoo? Intrările sunt adâncimea ciocului și lungimea aripii, standardizate pe setul de antrenare.

In [5]:
X_rest, X_test, s_rest, s_test = train_test_split(
    df[masuri], df["species"], test_size=0.15,
    stratify=df["species"], random_state=0)
X_ant, X_val, s_ant, s_val = train_test_split(
    X_rest, s_rest, test_size=0.15 / 0.85,
    stratify=s_rest, random_state=0)
print(len(X_ant), len(X_val), len(X_test))

238 52 52


In [6]:
def intrari(t, col):   # standardizate pe antrenare
    m, s = X_ant[col].mean(), X_ant[col].std()
    z = (t[col] - m) / s
    return torch.tensor(z.values).float()

def gentoo(s):   # 1 pentru Gentoo, altfel 0
    t = torch.tensor((s == "Gentoo").values)
    return t.float().reshape(-1, 1)

col = ["bill_depth_mm", "flipper_length_mm"]
z_ant = intrari(X_ant, col)
z_val = intrari(X_val, col)
t_ant, t_val = gentoo(s_ant), gentoo(s_val)
print(z_ant.shape, round(t_val.mean().item(), 3))

torch.Size([238, 2]) 0.365


Neuronul are o pondere pentru fiecare intrare și un termen liber. `nn.BCEWithLogitsLoss` face singură sigmoida și pierderea (§3). Bucla este cea din laboratorul 4.

In [7]:
torch.manual_seed(0)
# scorul = w1 × cioc + w2 × aripa + b
neuron = nn.Linear(2, 1)
antreneaza(neuron, nn.BCEWithLogitsLoss(),
           z_ant, t_ant, rata=0.5, epoci=50)
print(acuratete(neuron, z_val, t_val))
print(round(1 - t_val.mean().item(), 3))  # referința

1.0
0.635


**Perceptronul din cursul 2 și regresia logistică (§2.4).** Aceeași sumă ponderată; perceptronul răspunde cu treapta, regresia logistică cu o probabilitate. Pentru un neuron încă neantrenat, comparăm corecția lui Rosenblatt, (t − ŷ) · x, cu pasul dat de gradient, adică minus gradientul, (t − p) · x, pe un pinguin clasificat corect și pe unul clasificat greșit:

In [8]:
torch.manual_seed(0)
n0 = nn.Linear(2, 1)   # neantrenat
with torch.no_grad():
    bun = ((n0(z_ant) > 0).float() == t_ant).squeeze()
for caz, i in (("corect", int(bun.nonzero()[0])),
               ("greșit", int((~bun).nonzero()[0]))):
    x, t = z_ant[i:i+1], t_ant[i:i+1]
    n0.zero_grad()
    s = n0(x)
    y_cap = (s > 0).float()          # treapta
    nn.BCEWithLogitsLoss()(s, t).backward()
    p = torch.sigmoid(s).item()
    print(caz, "p =", round(p, 2))
    ros = ((t - y_cap) * x).squeeze()   # Rosenblatt
    grad = -n0.weight.grad.squeeze()    # gradientul
    print("  Rosenblatt:", ros.numpy().round(2))
    print("  gradient:  ", grad.numpy().round(2))

corect p = 0.54
  Rosenblatt: [0. 0.]
  gradient:   [0.03 0.91]
greșit p = 0.43
  Rosenblatt: [-1.95  0.74]
  gradient:   [-1.11  0.43]


### Notițele mele

…

## D2. Entropia încrucișată (§3)

Pierderea unui pinguin este −ln din probabilitatea pe care modelul a dat-o clasei corecte. Tabelul din cursul 3:

In [9]:
p = torch.tensor([0.99, 0.7, 0.5, 0.01])
print((-torch.log(p)).round(decimals=2))

tensor([0.0100, 0.3600, 0.6900, 4.6100])


### Încercați acum

Modelul dă unui pinguin probabilitatea 0,8 de a fi Gentoo. Calculați întâi pe hârtie, cu tabelul de mai sus: cât este pierderea dacă pinguinul este Gentoo? Dar dacă nu este? Apoi schimbați probabilitatea în celula de mai jos (de exemplu 0.99 sau 0.5).

In [10]:
p_gentoo = 0.8   # schimbați probabilitatea
p = torch.tensor(p_gentoo)
print(round(-torch.log(p).item(), 2))  # este Gentoo
print(round(-torch.log(1 - p).item(), 2))  # nu este

0.22
1.61


Cu trei specii, modelul dă trei scoruri. Softmax le transformă în trei probabilități pozitive, cu suma 1. Scorurile de mai jos dau chiar pinguinul Adelie din cursul 3.

In [11]:
# scorurile pentru Adelie, Chinstrap, Gentoo
scoruri = torch.tensor([1.95, 0.7, 0.0])
p = torch.softmax(scoruri, dim=0)
print(p.round(decimals=2))
print(round(-torch.log(p[0]).item(), 2))

tensor([0.7000, 0.2000, 0.1000])
0.36


Toate cele trei specii deodată: un strat cu trei ieșiri, pe trei măsurători. `nn.CrossEntropyLoss` face softmax și −ln într-un singur pas, ca la laboratorul 3; specia aleasă este cea cu scorul cel mai mare.

In [12]:
specii = ["Adelie", "Chinstrap", "Gentoo"]
z3_ant = intrari(X_ant, masuri[:3])
z3_val = intrari(X_val, masuri[:3])
c_ant = torch.tensor(s_ant.map(specii.index).values)
c_val = torch.tensor(s_val.map(specii.index).values)

torch.manual_seed(0)
strat = nn.Linear(3, 3)   # un scor pe specie
antreneaza(strat, nn.CrossEntropyLoss(),
           z3_ant, c_ant, rata=0.5, epoci=50)
with torch.no_grad():
    alese = strat(z3_val).argmax(dim=1)
corect = (alese == c_val).float().mean().item()
print(round(corect, 3))

0.981


**De ce nu pătratul erorii la clasificare (§3.4).** Pentru un pinguin Gentoo (eticheta 1), cât de tare împinge fiecare pierdere scorul, după ce sigmoida l-a transformat în probabilitate? Coloanele: scorul, probabilitatea, panta pierderii pătratice și panta entropiei încrucișate, ambele după scor.

In [13]:
t = torch.tensor([1.0])
for scor in (-4.0, 0.0, 4.0):
    # pătratul erorii, după sigmoidă
    s1 = torch.tensor([scor], requires_grad=True)
    p = torch.sigmoid(s1)
    ((p - t) ** 2).mean().backward()
    # entropia încrucișată
    s2 = torch.tensor([scor], requires_grad=True)
    nn.BCEWithLogitsLoss()(s2, t).backward()
    print(scor, round(p.item(), 3),
          round(s1.grad.item(), 3),
          round(s2.grad.item(), 3))

-4.0 0.018 -0.035 -0.982
0.0 0.5 -0.25 -0.5
4.0 0.982 -0.001 -0.018


### Notițele mele

…

## D3. Mascul sau femelă: un neuron și o rețea (§4)

Întrebarea nouă: este pinguinul mascul sau femelă, după masa corporală și lungimea ciocului? Folosim cei 333 de pinguini pentru care se știe, cu aceeași împărțire 70/15/15, stratificată pe specii. Eticheta este 1 pentru mascul și 0 pentru femelă.

In [14]:
mf = df.dropna(subset=["sex"])   # 333 de pinguini
M_rest, M_test = train_test_split(
    mf, test_size=0.15, stratify=mf["species"],
    random_state=0)
M_ant, M_val = train_test_split(
    M_rest, test_size=0.15 / 0.85,
    stratify=M_rest["species"], random_state=0)

col_mf = ["body_mass_g", "bill_length_mm"]
med, ab = M_ant[col_mf].mean(), M_ant[col_mf].std()

def standard(t):   # cu media și abaterea de antrenare
    z = (t[col_mf] - med) / ab
    return torch.tensor(z.values).float()

def mascul(t):     # 1 pentru mascul, 0 pentru femelă
    e = torch.tensor((t["sex"] == "MALE").values)
    return e.float().reshape(-1, 1)

zm_ant, zm_val = standard(M_ant), standard(M_val)
ym_ant, ym_val = mascul(M_ant), mascul(M_val)
print(len(M_ant), len(M_val), len(M_test))
# referința care spune mereu femelă
print(round(1 - ym_val.mean().item(), 3))

233 50 50
0.52


Întâi un singur neuron, ca la Gentoo: o dreaptă în planul celor două măsurători.

In [15]:
torch.manual_seed(0)
neuron_mf = nn.Linear(2, 1)
antreneaza(neuron_mf, nn.BCEWithLogitsLoss(),
           zm_ant, ym_ant, rata=0.1, epoci=300)
print(acuratete(neuron_mf, zm_ant, ym_ant),
      acuratete(neuron_mf, zm_val, ym_val))

0.601 0.66


**Comanda suspectă din cursul 2 (§4.1).** Aceeași limită, pe cele patru comenzi: suspectă dacă exact una dintre condiții este 1. Cu bucla din laboratorul 4 și entropia încrucișată, un neuron și o rețea cu 8 neuroni ascunși; afișăm probabilitatea „suspectă” pentru fiecare comandă:

In [16]:
X4 = torch.tensor([[0., 0.], [1., 0.],
                   [0., 1.], [1., 1.]])
XOR = torch.tensor([[0.], [1.], [1.], [0.]])
torch.manual_seed(0)
un_neuron = nn.Linear(2, 1)
antreneaza(un_neuron, nn.BCEWithLogitsLoss(),
           X4, XOR, rata=0.5, epoci=2000)
torch.manual_seed(0)
doua_straturi = nn.Sequential(
    nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1))
antreneaza(doua_straturi, nn.BCEWithLogitsLoss(),
           X4, XOR, rata=0.5, epoci=2000)
with torch.no_grad():
    for m in (un_neuron, doua_straturi):
        p = torch.sigmoid(m(X4)).squeeze()
        print(p.round(decimals=2))

tensor([0.5000, 0.5000, 0.5000, 0.5000])
tensor([0., 1., 1., 0.])


Între straturi punem o funcție neliniară. ReLU lasă să treacă valorile pozitive și pune 0 în locul celor negative:

In [17]:
print(torch.relu(torch.tensor([-2.0, 0.0, 3.0])))

tensor([0., 0., 3.])


Rețeaua: un strat ascuns cu 16 neuroni, ReLU, apoi un neuron de ieșire. Bucla, pierderea, rata și numărul de epoci sunt aceleași ca la neuron.

In [18]:
torch.manual_seed(0)
retea = nn.Sequential(
    nn.Linear(2, 16),   # stratul ascuns
    nn.ReLU(),
    nn.Linear(16, 1),   # neuronul de ieșire
)
antreneaza(retea, nn.BCEWithLogitsLoss(),
           zm_ant, ym_ant, rata=0.1, epoci=300)
print(acuratete(retea, zm_ant, ym_ant),
      acuratete(retea, zm_val, ym_val))
print(sum(p.numel() for p in retea.parameters()))

0.863 0.88
65


### Încercați acum

În perechi: câți neuroni ascunși îi trebuie rețelei? Prevedeți întâi: cu 2 neuroni ascunși, ajunge rețeaua la 88% pe validare? Dar cu 64? Apoi rulați și notați acuratețea pe antrenare și pe validare. Puteți încerca și alte numere.

In [19]:
def retea_cu(ascunsi):
    torch.manual_seed(0)
    r = nn.Sequential(nn.Linear(2, ascunsi),
                      nn.ReLU(),
                      nn.Linear(ascunsi, 1))
    antreneaza(r, nn.BCEWithLogitsLoss(),
               zm_ant, ym_ant, rata=0.1, epoci=300)
    return (acuratete(r, zm_ant, ym_ant),
            acuratete(r, zm_val, ym_val))

for ascunsi in (2, 4, 16, 64):   # schimbați lista
    print(ascunsi, retea_cu(ascunsi))

2 (0.695, 0.7)


4 (0.734, 0.72)


16 (0.863, 0.88)


64 (0.863, 0.88)


**Funcția de activare se schimbă dintr-un rând (§4.4).** Aceeași rețea, cu ReLU, tanh, sigmoida sau fără nicio funcție între straturi (`nn.Identity`), cu aceeași buclă, 300 de epoci; ultimul rând este sigmoida după 3.000 de epoci.

In [20]:
def cu_activare(activare, epoci=300):
    torch.manual_seed(0)
    r = nn.Sequential(nn.Linear(2, 16), activare(),
                      nn.Linear(16, 1))
    antreneaza(r, nn.BCEWithLogitsLoss(),
               zm_ant, ym_ant, rata=0.1, epoci=epoci)
    return (acuratete(r, zm_ant, ym_ant),
            acuratete(r, zm_val, ym_val))

activari = [("ReLU", nn.ReLU), ("tanh", nn.Tanh),
            ("sigmoida", nn.Sigmoid),
            ("fără", nn.Identity)]
for nume, activare in activari:
    print(nume, cu_activare(activare))
print("sigmoida, 3000", cu_activare(nn.Sigmoid, 3000))

ReLU (0.863, 0.88)


tanh (0.867, 0.88)


sigmoida (0.605, 0.64)


fără (0.605, 0.64)


sigmoida, 3000 (0.863, 0.9)


### Notițele mele

…

## D4. Propagarea înapoi (§5)

**Regula lanțului pe un drum, cu numere (§5.2).** Pentru un singur pinguin, gradientul ponderilor neuronului de ieșire se calculează de mână: (probabilitatea − eticheta) înmulțit cu ieșirile stratului ascuns. Este exact ce dă `backward()`:

In [21]:
z1, y1 = zm_ant[:1], ym_ant[:1]   # un pinguin
retea.zero_grad()
h = torch.relu(retea[0](z1))      # stratul ascuns
s = retea[2](h)                   # scorul
nn.BCEWithLogitsLoss()(s, y1).backward()
de_mana = (torch.sigmoid(s) - y1) * h
print(torch.allclose(de_mana, retea[2].weight.grad))

True


Un pas mai departe, în stratul ascuns: drumul de la pierdere la prima pondere a unui neuron ascuns trece prin scor, prin ponderea v a neuronului spre ieșire, prin ReLU și prin intrare. Pe un pinguin clasificat greșit de rețea, gradientul este produsul pantelor de pe drum: (p − y) × v × 1 × x₁ (panta lui ReLU este 1, pentru că neuronul este activ).

In [22]:
with torch.no_grad():
    gresit = ((retea(zm_ant) > 0).float() != ym_ant)
# primul pinguin clasificat greșit
i = int(gresit.squeeze().nonzero()[0])
z1, y1 = zm_ant[i:i+1], ym_ant[i:i+1]
retea.zero_grad()
a = retea[0](z1)                   # înainte de ReLU
s = retea[2](torch.relu(a))
nn.BCEWithLogitsLoss()(s, y1).backward()
p = torch.sigmoid(s).item()
j = int((a[0] > 0).nonzero()[0])   # un neuron activ
v = retea[2].weight[0, j].item()
x1 = z1[0, 0].item()
print("p =", round(p, 3), "y =", int(y1.item()),
      "v =", round(v, 3), "x1 =", round(x1, 3))
print(round((p - y1.item()) * v * 1 * x1, 4),
      round(retea[0].weight.grad[j, 0].item(), 4))

p = 0.649 y = 0 v = 2.114 x1 = -1.031
-1.4143 -1.4143


**Gradientul pe straturi (§5.3).** Pe o rețea nouă, înainte de antrenare, un singur `backward()` pe toți pinguinii de antrenare: fiecare strat își primește gradientul.

In [23]:
torch.manual_seed(0)
noua = nn.Sequential(nn.Linear(2, 16), nn.ReLU(),
                     nn.Linear(16, 1))
pierdere_f = nn.BCEWithLogitsLoss()
pierdere = pierdere_f(noua(zm_ant), ym_ant)
pierdere.backward()
for nume, p in noua.named_parameters():
    print(nume, list(p.grad.shape),
          round(p.grad.norm().item(), 3))

0.weight [16, 2] 0.06
0.bias [16] 0.03
2.weight [1, 16] 0.24
2.bias [1] 0.011


Unde ReLU a dat 0, gradientul nu trece mai departe. În rețeaua antrenată, câți dintre cei 16 neuroni ascunși dau 0 pentru toți pinguinii de antrenare?

In [24]:
with torch.no_grad():
    # ieșirile stratului ascuns: [233, 16]
    ascuns = torch.relu(retea[0](zm_ant))
mereu_zero = (ascuns == 0).all(dim=0)
print(ascuns.shape, int(mereu_zero.sum()))

torch.Size([233, 16]) 2


**De ce nu pornim de la zero (§5.4).** Aceeași rețea, pornită cu toți parametrii 0 și apoi cu toți parametrii 0,5, în locul pornirii aleatoare din PyTorch. Afișăm acuratețea pe antrenare și pe validare și câți neuroni ascunși diferiți are rețeaua după antrenare (rânduri diferite în ponderile stratului ascuns).

In [25]:
for start in (0.0, 0.5):
    torch.manual_seed(0)
    r = nn.Sequential(nn.Linear(2, 16), nn.ReLU(),
                      nn.Linear(16, 1))
    with torch.no_grad():
        for parametru in r.parameters():
            parametru.fill_(start)   # toți la fel
    antreneaza(r, nn.BCEWithLogitsLoss(),
               zm_ant, ym_ant, rata=0.1, epoci=300)
    w = r[0].weight.detach()   # stratul ascuns
    diferiti = len(torch.unique(w, dim=0))
    print(start, acuratete(r, zm_ant, ym_ant),
          acuratete(r, zm_val, ym_val), diferiti)

0.0 0.511 0.48 1


0.5 0.687 0.7 1


### Notițele mele

…

## Mai departe

La laboratorul 5 antrenați un neuron pentru Adelie și Chinstrap, apoi o rețea pentru întrebarea „mascul sau femelă?”, pe o altă pereche de măsurători, și alegeți câți neuroni ascunși îi trebuie. Funcția `pas` rămâne cea din laboratorul 4. La cursul 6: optimizatori mai buni decât SGD și curbele de antrenare și de validare. Caietul de rampă 3 („Probabilități și entropie încrucișată”) repetă §3, cu exerciții care se verifică singure.